In [21]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install -U transformers datasets fsspec
!pip install pandas scikit-learn



  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)


In [22]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os
import glob
import shutil
import zipfile
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from transformers.modeling_outputs import ModelOutput
from datasets import Dataset, ClassLabel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, mean_squared_error, mean_absolute_error, classification_report



In [24]:
# ============================================================
# CELL 3: Upload training data
# ============================================================
from google.colab import files

uploaded = files.upload()
zip_path = next(iter(uploaded))  # name of the uploaded zip file

extract_dir = "/content/training_data"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Extracted training data to {extract_dir}")



Saving dataset.zip to dataset (2).zip
Extracted training data to /content/training_data


In [25]:
# ============================================================
# CELL 4: View how data is spread across events
#
# Searches subfolders too (recursive=True) - zips usually extract
# into a nested folder rather than dropping CSVs at the top level.
# ============================================================
csv_files_preview = glob.glob(os.path.join(extract_dir, "**", "*.csv"), recursive=True)
if not csv_files_preview:
    raise FileNotFoundError(
        f"No .csv files found under {extract_dir}. Contents: {os.listdir(extract_dir)}"
    )

df_preview = pd.concat([pd.read_csv(f) for f in csv_files_preview], ignore_index=True)

event_col = "event_class" if "event_class" in df_preview.columns else "event"
print(df_preview[event_col].value_counts())
print(f"Total rows: {len(df_preview)}")



event_class
normal_play    2151
scrum           149
goal_kick       121
lineout         111
tmo_replay       62
penalty          54
kick_off         52
maul             44
try              40
Name: count, dtype: int64
Total rows: 2784


In [26]:

# ============================================================
# CELL 5: Build label maps
#
# normal_play is pinned to id 0 - CELL 11 defaults any blank or
# unrecognized event_class value to label 0, so 0 must stay
# normal_play regardless of where it sits in EVENT_CLASSES.
# ============================================================
EVENT_CLASSES = [
    "try",
    "goal_kick",
    "card_event",
    "scrum",
    "maul",
    "lineout",
    "kick_off",
    "tmo_replay",
    "penalty",
]

label2id = {"normal_play": 0}
id2label = {0: "normal_play"}
for i, evt in enumerate(EVENT_CLASSES, start=1):
    label2id[evt] = i
    id2label[i] = evt

num_labels = len(label2id)
print("Labels:", label2id)



Labels: {'normal_play': 0, 'try': 1, 'goal_kick': 2, 'card_event': 3, 'scrum': 4, 'maul': 5, 'lineout': 6, 'kick_off': 7, 'tmo_replay': 8, 'penalty': 9}


In [27]:

# ============================================================
# CELL 6: Model definition - DualHeadRoBERTa
#   - event_classifier: predicts the event type (try, penalty, ...)
#   - highlight_scorer: predicts a 0-1 "how exciting is this" score
# ============================================================
MODEL_NAME = "roberta-base"  # change if training a different base model


@dataclass
class DualHeadRoBERTaOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    highlight_scores: torch.FloatTensor = None


class DualHeadRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels, id2label, label2id):
        super().__init__()
        self.config = AutoModel.from_pretrained(model_name).config
        self.config.num_labels = num_labels
        self.config.id2label = id2label
        self.config.label2id = label2id

        self.roberta = AutoModel.from_pretrained(model_name)

        self.event_classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.highlight_scorer = nn.Linear(self.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, labels=None, highlight_labels=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = getattr(outputs, "pooler_output", None)
        if pooled_output is None:
            pooled_output = outputs.last_hidden_state[:, 0, :]

        event_logits = self.event_classifier(pooled_output)
        highlight_score = torch.sigmoid(self.highlight_scorer(pooled_output))

        return DualHeadRoBERTaOutput(logits=event_logits, highlight_scores=highlight_score)



In [28]:

# ============================================================
# CELL 7: Loss functions
#
# FocalLoss: for the event classifier - handles class imbalance
#   automatically, so normal_play should NOT be down-weighted
#   separately (that destroys precision, see class_weights below).
# WeightedMSELoss: for the highlight scorer - upweights samples
#   with a meaningfully high highlight score.
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


class WeightedMSELoss(nn.Module):
    def __init__(self, base_weight=1.0, high_score_weight=5.0, threshold=0.01):
        super().__init__()
        self.base_weight = base_weight
        self.high_score_weight = high_score_weight
        self.threshold = threshold

    def forward(self, inputs, targets):
        mse = F.mse_loss(inputs, targets, reduction="none")
        weights = torch.where(targets > self.threshold, self.high_score_weight, self.base_weight)
        return (mse * weights).mean()



In [29]:

# ============================================================
# CELL 8: Custom Trainer
#
# stage=1 trains the event classifier head, stage=2 trains the
# highlight scorer head (see CELL 13/15).
# ============================================================
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, stage=1, **kwargs):
        super().__init__(*args, **kwargs)
        self.stage = stage
        self.loss_fct_events = FocalLoss(alpha=class_weights, gamma=2.0)
        self.loss_fct_highlights = WeightedMSELoss(base_weight=1.0, high_score_weight=5.0, threshold=0.01)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        highlight_labels = inputs.pop("highlight_labels")

        outputs = model(**inputs)
        logits = outputs.logits
        highlight_scores = outputs.highlight_scores

        if self.stage == 1:
            if self.loss_fct_events.alpha is not None and self.loss_fct_events.alpha.device != logits.device:
                self.loss_fct_events.alpha = self.loss_fct_events.alpha.to(logits.device)
            loss = self.loss_fct_events(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        else:
            loss = self.loss_fct_highlights(highlight_scores.view(-1), highlight_labels.view(-1).to(highlight_scores.dtype))

        return (loss, outputs) if return_outputs else loss


In [30]:

# ============================================================
# CELL 9: Evaluation metrics
#
# The model has two label columns (labels, highlight_labels), so
# Trainer passes both heads' predictions/targets here once per
# evaluation pass (each epoch, since eval_strategy="epoch"). This
# reports classification metrics for the event head AND regression
# metrics (mse, mae) for the highlight head, in every epoch's
# validation row - even though only one head is actively training
# per stage.
#
# Both weighted and macro averages are reported for precision/
# recall/f1. Weighted is dominated by normal_play (the majority
# class), so a model can look good on it while doing badly on rare
# events like try/maul. Macro treats every class equally, which is
# why CELL 13 selects the best Stage 1 checkpoint by f1_macro
# instead of the weighted f1.
# ============================================================
def compute_metrics(eval_pred):
    (event_logits, highlight_preds) = eval_pred.predictions
    (event_labels, highlight_labels) = eval_pred.label_ids

    event_preds = event_logits.argmax(axis=-1)
    accuracy = accuracy_score(event_labels, event_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        event_labels, event_preds, average="weighted", zero_division=0
    )
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        event_labels, event_preds, average="macro", zero_division=0
    )

    highlight_preds = highlight_preds.reshape(-1)
    highlight_labels = highlight_labels.reshape(-1)
    mse = mean_squared_error(highlight_labels, highlight_preds)
    mae = mean_absolute_error(highlight_labels, highlight_preds)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "mse": mse,
        "mae": mae,
    }



In [31]:
# ============================================================
# CELL 10: Load the tokenizer + model (fresh start every run)
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DualHeadRoBERTa(MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:

# ============================================================
# CELL 11: Load CSVs, build sliding-window text, and compute class weights
#
# WINDOW_SIZE=5 concatenates each row with its 2 neighbors on either
# side into one training sample, and uses the CENTER row's label and
# highlight_score as the target. Commentary describing an event (e.g.
# a try) usually builds across several consecutive lines, so this
# gives the model that surrounding context instead of judging one
# isolated line. MAX_LENGTH is bumped 128 -> 256 (CELL 12) to fit the
# longer windowed text.
#
# Windowing runs per CSV file, BEFORE concatenating them together, so
# a window can never bridge two different uploaded matches. The first
# and last WINDOW_SIZE//2 rows of each file are dropped rather than
# padded with a shorter window, so every sample has full context.
#
# Note for later: any inference code must build the same 5-line
# window around a line before classifying it - feeding a single raw
# line to this model would not match what it was trained on.
# ============================================================
WINDOW_SIZE = 5
MAX_LENGTH = 256
window_half = WINDOW_SIZE // 2

csv_files = glob.glob(os.path.join(extract_dir, "**", "*.csv"), recursive=True)
windowed_dfs = []
for f in csv_files:
    d = pd.read_csv(f)
    if "event_class" in d.columns:
        d["label"] = d["event_class"].apply(lambda x: label2id.get(str(x).strip(), 0) if pd.notna(x) else 0)
    elif "event" in d.columns:
        d["label"] = d["event"].apply(lambda x: label2id.get(str(x).strip(), 0) if pd.notna(x) else 0)

    if "highlight_score" in d.columns:
        d["highlight_label"] = pd.to_numeric(d["highlight_score"], errors="coerce").fillna(0.0)
    else:
        d["highlight_label"] = 0.0

    d = d.dropna(subset=["text"]).reset_index(drop=True)
    texts = d["text"].tolist()

    if len(texts) <= 2 * window_half:
        print(f"Skipping {f}: only {len(texts)} rows, need more than {2 * window_half} for window size {WINDOW_SIZE}")
        continue

    rows = []
    for i in range(window_half, len(texts) - window_half):
        window_text = " ".join(texts[i - window_half : i + window_half + 1])
        rows.append({
            "text": window_text,
            "label": d.loc[i, "label"],
            "highlight_label": d.loc[i, "highlight_label"],
        })
    windowed_dfs.append(pd.DataFrame(rows))

df = pd.concat(windowed_dfs, ignore_index=True)
df_balanced = df.sample(frac=1, random_state=42).reset_index(drop=True)

total_samples = len(df_balanced)
class_counts = df_balanced["label"].value_counts().to_dict()
weights = []
for i in range(num_labels):
    count = class_counts.get(i, 0)
    w = total_samples / (num_labels * count) if count > 0 else 1.0
    weights.append(w)

class_weights = torch.tensor(weights, dtype=torch.float)
print("Class weights:", class_weights)



Class weights: tensor([0.1294, 6.9500, 2.2975, 1.0000, 1.8658, 6.3182, 2.5045, 5.5600, 4.4839,
        5.1481])


In [33]:

# ============================================================
# CELL 12: Tokenize and split into train/test
#
# stratify_by_column="label" keeps every event class represented in
# both the train and test split, proportional to its overall
# frequency. Without it, a plain random split can leave a rare class
# (e.g. try, maul) with zero test rows - which is exactly what broke
# CELL 14's classification_report earlier. train_test_split requires
# the stratify column to be a ClassLabel feature, hence the cast.
# ============================================================
dataset = Dataset.from_pandas(df_balanced)
dataset = dataset.cast_column(
    "label", ClassLabel(num_classes=num_labels, names=[id2label[i] for i in range(num_labels)])
)


def tokenize_func(examples):
    tokenized = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)
    tokenized["labels"] = examples["label"]
    tokenized["highlight_labels"] = examples["highlight_label"]
    return tokenized


tokenized_datasets = dataset.map(tokenize_func, batched=True)
split_datasets = tokenized_datasets.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")

OUTPUT_DIR = "/content/model_output"


def get_training_args(output_dir_suffix, epochs=5, metric_for_best_model="loss", greater_is_better=None):
    return TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/{output_dir_suffix}",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        logging_strategy="steps",
        logging_steps=5,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model=metric_for_best_model,
        greater_is_better=greater_is_better,
        learning_rate=2e-5,
        warmup_ratio=0.1,
    )



Casting the dataset:   0%|          | 0/2780 [00:00<?, ? examples/s]

Map:   0%|          | 0/2780 [00:00<?, ? examples/s]

In [34]:

# ============================================================
# CELL 13: Stage 1 - Train the event classifier
#
# metric_for_best_model="f1_macro" makes load_best_model_at_end
# restore the epoch that was actually best at classifying events
# (weighted across all classes equally), not just the epoch with
# the lowest FocalLoss - so `model` holds the best checkpoint's
# weights, not simply the last epoch's, once training finishes.
# ============================================================
trainer_stage1 = CustomTrainer(
    model=model,
    args=get_training_args("stage1", epochs=10, metric_for_best_model="f1_macro", greater_is_better=True),
    train_dataset=split_datasets["train"],
    eval_dataset=split_datasets["test"],
    class_weights=class_weights,
    stage=1,
    compute_metrics=compute_metrics,
)
trainer_stage1.train()




[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Precision Macro,Recall Macro,F1 Macro,Mse,Mae
1,2.625783,1.494810,0.082734,0.024636,0.082734,0.029813,0.105724,0.306061,0.120299,0.108652,0.270644
2,0.589183,0.532918,0.188849,0.069847,0.188849,0.092939,0.252871,0.751178,0.343487,0.141438,0.325794
3,0.177697,0.181740,0.399281,0.822976,0.399281,0.412616,0.406358,0.863838,0.466584,0.113296,0.279275
4,0.165915,0.264483,0.631295,0.861393,0.631295,0.683185,0.564820,0.848003,0.614188,0.134080,0.313689
5,0.192994,0.149680,0.863309,0.910311,0.863309,0.875644,0.712539,0.927322,0.796161,0.117135,0.284979
6,0.001176,0.139977,0.850719,0.908082,0.850719,0.865535,0.681966,0.915328,0.764597,0.116903,0.284879
7,0.003316,0.154454,0.890288,0.922472,0.890288,0.898273,0.726895,0.941939,0.810933,0.123246,0.295493
8,0.001462,0.147459,0.908273,0.927953,0.908273,0.913282,0.746081,0.933782,0.823647,0.118211,0.287612
9,0.001279,0.162542,0.911871,0.930311,0.911871,0.916253,0.757669,0.937744,0.833578,0.119748,0.290383
10,0.001437,0.154360,0.919065,0.934234,0.919065,0.922767,0.763234,0.938778,0.838261,0.119704,0.290216


TrainOutput(global_step=1390, training_loss=0.2997256365482421, metrics={'train_runtime': 1291.3924, 'train_samples_per_second': 17.222, 'train_steps_per_second': 1.076, 'total_flos': 0.0, 'train_loss': 0.2997256365482421, 'epoch': 10.0})

In [35]:

# ============================================================
# CELL 14: Per-class classification report (event head)
#
# compute_metrics only reports weighted-average precision/recall/f1
# each epoch. This runs one prediction pass over the held-out test
# set and prints sklearn's full per-class breakdown, like:
#   precision  recall  f1-score  support   (one row per event)
# ============================================================
pred_output = trainer_stage1.predict(split_datasets["test"])
event_logits, _ = pred_output.predictions
event_labels, _ = pred_output.label_ids
event_preds = event_logits.argmax(axis=-1)

target_names = [id2label[i] for i in range(num_labels)]
print(classification_report(
    event_labels, event_preds, labels=list(range(num_labels)), target_names=target_names, zero_division=0
))



              precision    recall  f1-score   support

 normal_play       0.99      0.91      0.95       430
         try       0.67      1.00      0.80         8
   goal_kick       0.89      1.00      0.94        24
  card_event       0.00      0.00      0.00         0
       scrum       0.76      0.97      0.85        30
        maul       0.73      0.89      0.80         9
     lineout       0.68      0.86      0.76        22
    kick_off       0.71      1.00      0.83        10
  tmo_replay       0.80      1.00      0.89        12
     penalty       0.64      0.82      0.72        11

    accuracy                           0.92       556
   macro avg       0.69      0.84      0.75       556
weighted avg       0.93      0.92      0.92       556



In [36]:

# ============================================================
# CELL 15: Stage 2 - Train the highlight scorer
# (backbone + event head are frozen so stage 1 knowledge is kept)
#
# metric_for_best_model="mse" + greater_is_better=False restores the
# epoch with the lowest highlight-score error at the end of training,
# same reasoning as Stage 1's f1_macro choice.
# ============================================================
for name, param in model.named_parameters():
    if "highlight_scorer" not in name:
        param.requires_grad = False

trainer_stage2 = CustomTrainer(
    model=model,
    args=get_training_args("stage2", epochs=5, metric_for_best_model="mse", greater_is_better=False),
    train_dataset=split_datasets["train"],
    eval_dataset=split_datasets["test"],
    class_weights=class_weights,
    stage=2,
    compute_metrics=compute_metrics,
)
trainer_stage2.train()

for param in model.parameters():
    param.requires_grad = True



[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Precision Macro,Recall Macro,F1 Macro,Mse,Mae
1,0.219003,0.159298,0.919065,0.934234,0.919065,0.922767,0.763234,0.938778,0.838261,0.080692,0.223287
2,0.147510,0.131532,0.919065,0.934234,0.919065,0.922767,0.763234,0.938778,0.838261,0.063426,0.208426
3,0.101092,0.121460,0.919065,0.934234,0.919065,0.922767,0.763234,0.938778,0.838261,0.059077,0.205015
4,0.108876,0.116729,0.919065,0.934234,0.919065,0.922767,0.763234,0.938778,0.838261,0.057544,0.203505
5,0.112878,0.115314,0.919065,0.934234,0.919065,0.922767,0.763234,0.938778,0.838261,0.057175,0.203086


In [37]:

# ============================================================
# CELL 16: Save the best model and zip it
#
# `model` already holds the best checkpoint's weights, not the last
# epoch's: load_best_model_at_end (CELL 12) restores the best-f1_macro
# epoch after trainer_stage1.train() and the best-mse epoch after
# trainer_stage2.train(). So this is already saving the best event
# classifier + best highlight scorer, combined into one model.
# ============================================================
best_dir = os.path.join(OUTPUT_DIR, "best")
os.makedirs(best_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(best_dir, "pytorch_model.bin"))
model.config.save_pretrained(best_dir)
tokenizer.save_pretrained(best_dir)

out_zip_base = "/content/trained_model"
shutil.make_archive(out_zip_base, "zip", best_dir)
print(f"Model zipped to {out_zip_base}.zip")


Model zipped to /content/trained_model.zip


In [38]:

# ============================================================
# CELL 17: Download the trained model to your local machine
# ============================================================
files.download(f"{out_zip_base}.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>